# How Many Aquariums Are in the Dataset?

This notebook isolates **aquariums specifically** (excluding zoos and wildlife preserves) and reports the total count.

Self-contained: only requires `museums.csv` in the same folder. Run **Kernel → Restart Kernel and Run All Cells**.

## Setup

In [1]:
import pandas as pd
df = pd.read_csv("museums.csv", low_memory=False)
print(f"Loaded {len(df):,} museums")

Loaded 33,072 museums


## Step 1: Why we need both a category filter and a name filter

The IMLS dataset bundles zoos, aquariums, and wildlife conservation into one combined `Museum Type` category — there's no "aquarium" tag on its own. To isolate aquariums, we'll combine two filters:

1. The IMLS category (`ZOO, AQUARIUM, OR WILDLIFE CONSERVATION`) — narrows to 564 records.
2. A name match for `aquarium` — keeps only records whose name actually contains "aquarium."

## Step 2: Filter by category

In [2]:
category_mask = df["Museum Type"] == "ZOO, AQUARIUM, OR WILDLIFE CONSERVATION"
category_subset = df[category_mask]
print(f"Museums in the ZOO/AQUARIUM/WILDLIFE category: {len(category_subset)}")

Museums in the ZOO/AQUARIUM/WILDLIFE category: 564


## Step 3: Filter by name to isolate aquariums

In [3]:
name_mask = category_subset["Museum Name"].fillna("").str.contains(
    r"aquarium", case=False, regex=True)

aquariums = category_subset[name_mask]
print(f"Aquariums (in category AND name contains 'aquarium'): {len(aquariums)}")

Aquariums (in category AND name contains 'aquarium'): 100


### Spot-check: sample of matched aquariums

In [4]:
aquariums["Museum Name"].sample(15, random_state=1).tolist()

['TENNESSEE AQUARIUM',
 'DALLAS MARINE AQUARIUM SOCIETY',
 'SHEDD AQUARIUM SOCIETY',
 'AQUARIUM & RAINFOREST AT MOODY GARDENS',
 'TEXAS STATE AQUARIUM',
 'CLEARWATER MARINE AQUARIUM',
 'NEW ENGLAND AQUARIUM MARINE LIFE CENTER',
 'AQUARIUM AT ROCKPORT HARBOR',
 'MARK O. HATFIELD MARINE SCIENCE CENTER AQUARIUM NEWPORT',
 'COLUMBUS ZOO AND AQUARIUM',
 'SOUTH PADRE ISLAND AQUARIUM',
 'GREAT LAKES AQUARIUM',
 'SHARK REEF AQUARIUM AT MANDALAY BAY',
 'NORTH CAROLINA AQUARIUM SOCIETY',
 'NORTH CAROLINA AQUARIUM ON ROANOKE ISLAND']

## Step 4: A wrinkle — aquarium *societies*

Looking at the sample above, you'll notice entries like *"Greater Detroit Aquarium Society"* and *"San Francisco Aquarium Society."* These are hobbyist fish-keeping clubs, not public aquariums you'd visit. Whether they count as aquariums depends on your definition.

In [5]:
society_mask = aquariums["Museum Name"].str.contains(r"\bsociet", case=False, regex=True)
societies = aquariums[society_mask]
public_aquariums = aquariums[~society_mask]

print(f"All aquariums                        : {len(aquariums)}")
print(f"  ...hobbyist societies              : {len(societies)}")
print(f"  ...public aquariums (the rest)     : {len(public_aquariums)}")

print("\nExamples of hobbyist societies:")
for n in societies["Museum Name"].head(5):
    print(f"  • {n}")

print("\nExamples of public aquariums:")
for n in public_aquariums["Museum Name"].head(5):
    print(f"  • {n}")

All aquariums                        : 100
  ...hobbyist societies              : 30
  ...public aquariums (the rest)     : 70

Examples of hobbyist societies:
  • CENTRAL COAST AQUARIUM SOCIETY
  • MARINE AQUARIUM SOCIETY OF LOS ANGELES COUNTY
  • SAN DIEGO MARINE AQUARIUM SOCIETY
  • SAN FRANCISCO AQUARIUM SOCIETY
  • COLORADO AQUARIUM SOCIETY

Examples of public aquariums:
  • SONORAN SEA AQUARIUM
  • AQUARIUM OF THE BAY
  • AQUARIUM OF THE PACIFIC
  • CABRILLO MARINE AQUARIUM
  • MARINE SCIENCE DEPARTMENT PUBLIC AQUARIUM


## Step 5: Final answer

In [6]:
total = len(aquariums)
public = len(public_aquariums)

print("=" * 60)
print(f"  TOTAL AQUARIUMS IN THE DATASET:        {total}")
print(f"  ...excluding hobbyist societies:       {public}")
print("=" * 60)

  TOTAL AQUARIUMS IN THE DATASET:        100
  ...excluding hobbyist societies:       70


## Summary

**There are 100 aquariums in the dataset.**

This count comes from intersecting two filters:
- IMLS classifies the museum as `ZOO, AQUARIUM, OR WILDLIFE CONSERVATION` (564 records)
- The museum's name contains the word "aquarium" (case-insensitive)

**A stricter count: 70 public aquariums.** The 100 figure includes 30 entries that are actually aquarium *societies* — local hobbyist clubs for fish keepers — rather than public aquariums. If you only want institutions where visitors view fish, the answer is 70.

Same methodology as the zoos notebook: intersection of structured category + name pattern, which gives the tightest defensible count by excluding aquariums miscategorized in other buckets (e.g., 9 named "aquarium" but tagged as Science & Technology Museum).